In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [2]:
import os

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
# -------------------------
# Configure Gemini API key
# -------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [5]:
# -------------------------
# Create embedding model
# -------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_639/3271010774.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:


# -------------------------
# Create documents
# -------------------------

documents = [
    Document(page_content="Tesla is a company that manufactures electric vehicles."),
    Document(page_content="BMW is a German automobile manufacturer known for luxury cars."),
    Document(page_content="Toyota is famous for hybrid cars like the Prius."),
    Document(page_content="Nikola Tesla was a Serbian-American inventor known for AC electricity."),
]

# -------------------------
# Create vector database
# -------------------------

vector_db = FAISS.from_documents(documents, embedding_model)

In [7]:
# -------------------------
# Create Gemini LLM
# -------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [8]:


# -------------------------
# RAG prompt
# -------------------------

prompt_template = ChatPromptTemplate.from_template(
"""
Answer the question using the context below.

Context:
{context}

Question:
{question}
"""
)


In [11]:

# -------------------------
# User query
# -------------------------

query = input("Ask a question: ")

# Step 1: semantic search
retrieved_docs = vector_db.similarity_search(query, k=2)

context = "\n".join([doc.page_content for doc in retrieved_docs])

# Step 2: generate answer
prompt = prompt_template.format(
    context=context,
    question=query
)

response = llm.invoke(prompt)

Ask a question: What is the contribution of Tesla to electrical engineering


In [12]:
# -------------------------
# Output
# -------------------------

print("\nRetrieved Context:\n", context)
print("\nAnswer:\n", response.content)


Retrieved Context:
 Tesla is a company that manufactures electric vehicles.
Nikola Tesla was a Serbian-American inventor known for AC electricity.

Answer:
 According to the context, **Nikola Tesla** (the inventor) was known for **AC electricity**, which is a significant contribution to electrical engineering.
